In [ ]:
from nnsight import CONFIG

CONFIG.API.APIKEY = input("Enter your API key: ")
## list of remotely hosted models available here: https://nnsight.net/status/

# NDIF
NDIF (National Deep Inference Fabric) is a platform developed by the NSF in collaboration with Northeastern University for hosting frontier open-weight LLMs so US-based academic researchers can access them, even if they do not have local GPU resources. We'll be using NDIF to replicate some of the results in "Scaling Monosemanticity" on Eleuther AI's Pythia Language model. 

First, a brief tutorial on how to load a model on NDIF. We'll load deepseek, but there are many many other LLMs available. You can view the list here: https://nnsight.net/status/


In [2]:
from nnsight import LanguageModel
llm = LanguageModel(
    "deepseek-ai/DeepSeek-V3",
    device_map="auto",
    trust_remote_code=True
)
print(llm)

DeepseekV3ForCausalLM(
  (model): DeepseekV3Model(
    (embed_tokens): Embedding(129280, 7168)
    (layers): ModuleList(
      (0-2): 3 x DeepseekV3DecoderLayer(
        (self_attn): DeepseekV3Attention(
          (q_a_proj): Linear(in_features=7168, out_features=1536, bias=False)
          (q_a_layernorm): DeepseekV3RMSNorm()
          (q_b_proj): Linear(in_features=1536, out_features=24576, bias=False)
          (kv_a_proj_with_mqa): Linear(in_features=7168, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV3RMSNorm()
          (kv_b_proj): Linear(in_features=512, out_features=32768, bias=False)
          (o_proj): Linear(in_features=16384, out_features=7168, bias=False)
          (rotary_emb): DeepseekV3YarnRotaryEmbedding()
        )
        (mlp): DeepseekV3MLP(
          (gate_proj): Linear(in_features=7168, out_features=18432, bias=False)
          (up_proj): Linear(in_features=7168, out_features=18432, bias=False)
          (down_proj): Linear(in_features=18432

This is the deepseek model. There is a pretrained SAE available for Deepseek reasoning models trained by Goodfire (https://www.goodfire.ai/research/under-the-hood-of-a-reasoning-model#), but it is not compatible with NDIF. So we are going to use the pretrained SAE for Eleuther AI's Pythia 70M model

In [8]:
from nnsight import LanguageModel
from dictionary_learning.dictionary_learning.dictionary import AutoEncoder
import torch
from IPython.display import clear_output

In [ ]:
# Load pretrained autoencoder
# !./pretrained_dictionary_downloader.sh
# clear_output()

weights_path = "./dictionaries/pythia-70m-deduped/mlp_out_layer0/10_32768/ae.pt"
activation_dim = 512 # dimension of the NN's activations to be autoencoded
dictionary_size = 64 * activation_dim # number of features in the dictionary

ae = AutoEncoder(activation_dim, dictionary_size)
ae.load_state_dict(torch.load(weights_path,weights_only=True, map_location='cpu'))
# ae.cuda() # enable for cuda-capable systems


<All keys matched successfully>

# Applying SAE to Pythia70M

In [13]:
# now you should read through the scaling monosemanticity whitepaper and see if you can replicate some of hteir findings on another open source modle
model = LanguageModel("EleutherAI/pythia-70m-deduped", device_map="auto")
tokenizer = model.tokenizer

prompt = """
Call me Ishmael. Some years ago--never mind how long precisely--having little or no money in my purse, and nothing particular to interest me on shore, I thought I would sail about a little and see the watery part of the world. It is a way I have of driving off the spleen and regulating the circulation. Whenever I find myself growing grim about the mouth; whenever it is a damp, drizzly November in my soul; whenever I find myself involuntarily pausing before coffin warehouses, and bringing up the rear of every funeral I meet; and especially whenever my hypos get such an upper hand of me, that it requires a strong moral principle to prevent me from deliberately stepping into the street, and methodically knocking people's hats off--then, I account it high time to get to sea as soon as I can.
"""

# Extract layer 0 MLP output from base model
with model.trace(prompt) as tracer:
    mlp_0 = model.gpt_neox.layers[0].mlp.output.save()

# Use SAE to get features from activations
features = ae.encode(mlp_0)

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

In [14]:
# Find top features using the autoencoder
summed_activations = features.abs().sum(dim=1) # Sort by max activations
top_activations_indices = summed_activations.topk(20).indices # Get indices of top 20

compounded = []
for i in top_activations_indices[0]:
    compounded.append(features[:,:,i.item()].cpu()[0])

compounded = torch.stack(compounded, dim=0)

# Visualizing Features

In [15]:

from circuitsvis.tokens import colored_tokens_multi

tokens = tokenizer.encode(prompt)
str_tokens = [tokenizer.decode(t) for t in tokens]

# Visualize activations for top 20 most prominent features
colored_tokens_multi(str_tokens, compounded.T)

# Isolating Interpretable Features

1. Decide what feature you want to isolate (I want the features for Paris, France and Barack Obama)
2. Generate targeted positive and negative prompts that will/will not activate the feature (if it exists)
3. Run all examples (positive and negative examples) through the model and encode with the SAE
4. identify candidate features using $\texttt{(mean positive activation)} - \texttt{(mean negative activation)}$ as your ranking objective
5. From list of candidate features, refine selections by computing sensitivity, specificity, performing causal interventions 

2. Generate positive and negative prompts for feature

In [ ]:
import json
paris_path = "./data/paris_feature_dataset.json"
with open(paris_path, 'r') as f:
    paris = json.load(f)
print(f"json file fields: {paris.keys()}")

positive_examples_paris = paris['positive_examples']
negative_examples_paris = paris['negative_examples']
print("="*80)
for i, entry in enumerate(positive_examples_paris):
    print(f"{i}th entry is {entry}")

# how do I merge positive and negative examples into one big dict? 

json file fields: dict_keys(['description', 'target_token', 'positive_examples', 'negative_examples'])
0th entry is The Eiffel Tower is the most iconic landmark in Paris.
1th entry is Paris is the capital and largest city of France.
2th entry is She booked a one-way ticket to Paris for the spring.
3th entry is The Louvre Museum in Paris houses the Mona Lisa.
4th entry is Paris has been a global center for art and fashion for centuries.
5th entry is He proposed to her on a bridge over the Seine in Paris.
6th entry is The 2024 Summer Olympics were held in Paris.
7th entry is Notre-Dame Cathedral in Paris suffered a devastating fire in 2019.
8th entry is Paris is divided into twenty administrative districts called arrondissements.
9th entry is The metro system in Paris is one of the busiest in Europe.
10th entry is Hemingway once wrote that Paris is a moveable feast.
11th entry is The Treaty of Paris ended the American Revolutionary War in 1783.
12th entry is Paris has a population of ove

3. Run all examples through model and record activations

In [34]:
# load model 
model = LanguageModel("EleutherAI/pythia-70m-deduped", device_map="auto")
tokenizer = model.tokenizer

activation_dim = 512 # dimension of the NN's activations to be autoencoded
dictionary_size = 64 * activation_dim # number of features in the dictionary

# load one autoencoder per layer
target_layers = [0, 1, 2, 3, 4, 5]
autoencoders = {}
for layer in target_layers:
    weights_path = f"./dictionaries/pythia-70m-deduped/mlp_out_layer{layer}/10_32768/ae.pt"
    ae_layer = AutoEncoder(activation_dim, dictionary_size)
    ae_layer.load_state_dict(torch.load(weights_path, weights_only=True, map_location='cpu'))
    autoencoders[layer] = ae_layer

# initialize feature dict[key=prompt_set_promptidx_layer]: value=activations
features_dict = {}

# iterate over both positive and negative examples
for prompt_set in ['positive', 'negative']:

    # fetch pos / neg examples
    examples_key = f"{prompt_set}_examples"
    examples_subset = paris[examples_key]

    # for each prompt in pos/neg examples:
    for i, prompt in enumerate(examples_subset):
        # forward pass on prompt
        saved_mlps = {}
        with model.trace(prompt) as tracer:
            # collect activations for each layer
            for layer in target_layers:
                print(f"starting forward pass for prompt {i} ({prompt_set} features), layer {layer}")
                saved_mlps[layer] = model.gpt_neox.layers[layer].mlp.output.save()

        # encode with the correct SAE per layer (after trace exits)
        for layer in target_layers:
            features = autoencoders[layer].encode(saved_mlps[layer])
            key = f"{prompt_set}_{i}_{layer}"
            features_dict[key] = {
                'prompt': prompt,
                'features': features
            }

    print(f"finished {prompt_set} examples")

print(f"saved {len(features_dict)} entries")

starting forward pass for prompt 0 (positive features), layer 0
starting forward pass for prompt 0 (positive features), layer 1
starting forward pass for prompt 0 (positive features), layer 2
starting forward pass for prompt 0 (positive features), layer 3
starting forward pass for prompt 0 (positive features), layer 4
starting forward pass for prompt 0 (positive features), layer 5
starting forward pass for prompt 1 (positive features), layer 0
starting forward pass for prompt 1 (positive features), layer 1
starting forward pass for prompt 1 (positive features), layer 2
starting forward pass for prompt 1 (positive features), layer 3
starting forward pass for prompt 1 (positive features), layer 4
starting forward pass for prompt 1 (positive features), layer 5
starting forward pass for prompt 2 (positive features), layer 0
starting forward pass for prompt 2 (positive features), layer 1
starting forward pass for prompt 2 (positive features), layer 2
starting forward pass for prompt 2 (posi

In [35]:
# Compute per-feature mean activations for positive vs negative prompts, grouped by layer
from collections import defaultdict

pos_by_layer = defaultdict(list)
neg_by_layer = defaultdict(list)

for key, val in features_dict.items():
    encoded = val['features']  # already SAE-encoded: shape (1, seq_len, dictionary_size)
    mean_act = encoded.mean(dim=1).squeeze(0)  # mean over sequence positions

    # key format: "positive_0_3" -> prompt_set, prompt_idx, layer
    layer = int(key.rsplit('_', 1)[1])

    if key.startswith('positive'):
        pos_by_layer[layer].append(mean_act)
    else:
        neg_by_layer[layer].append(mean_act)

# Collect top features per layer, then rank globally
all_candidates = []

for layer in sorted(pos_by_layer.keys()):
    pos_mean = torch.stack(pos_by_layer[layer]).mean(dim=0)
    neg_mean = torch.stack(neg_by_layer[layer]).mean(dim=0)
    diff = pos_mean - neg_mean
    top_vals, top_idxs = diff.topk(20)
    for idx, val in zip(top_idxs, top_vals):
        all_candidates.append((val.item(), idx.item(), layer, pos_mean[idx].item(), neg_mean[idx].item()))

all_candidates.sort(key=lambda x: x[0], reverse=True)

print("Top 20 features by (mean positive - mean negative) activation:")
for rank, (diff_val, feat_idx, layer, pos_val, neg_val) in enumerate(all_candidates[:20]):
    print(f"  {rank+1:>2d}. Layer {layer}  Feature {feat_idx:>5d}  diff = {diff_val:.4f}  (pos = {pos_val:.4f}, neg = {neg_val:.4f})")

Top 20 features by (mean positive - mean negative) activation:
   1. Layer 0  Feature  6429  diff = 0.4667  (pos = 0.4689, neg = 0.0022)
   2. Layer 1  Feature 25874  diff = 0.0931  (pos = 0.1176, neg = 0.0245)
   3. Layer 4  Feature 17275  diff = 0.0919  (pos = 0.0987, neg = 0.0067)
   4. Layer 5  Feature   264  diff = 0.0858  (pos = 0.0869, neg = 0.0012)
   5. Layer 4  Feature  4255  diff = 0.0436  (pos = 0.0479, neg = 0.0043)
   6. Layer 3  Feature 10045  diff = 0.0428  (pos = 0.0532, neg = 0.0104)
   7. Layer 5  Feature 29320  diff = 0.0388  (pos = 0.5343, neg = 0.4956)
   8. Layer 5  Feature 26357  diff = 0.0387  (pos = 0.7389, neg = 0.7002)
   9. Layer 5  Feature  4424  diff = 0.0385  (pos = 0.0404, neg = 0.0019)
  10. Layer 5  Feature  6850  diff = 0.0358  (pos = 1.0879, neg = 1.0520)
  11. Layer 5  Feature 15388  diff = 0.0329  (pos = 0.7058, neg = 0.6730)
  12. Layer 3  Feature 16362  diff = 0.0327  (pos = 0.0329, neg = 0.0002)
  13. Layer 2  Feature  8314  diff = 0.0323  (pos

## Visualization 1: Max-Activating Tokens for Feature 6429

To find what tokens most strongly activate a feature, we run a large corpus (the Pile 10k dataset) through the model and record Feature 6429's activation at every token position. We then rank all tokens by activation strength. This is the standard approach used by tools like Neuronpedia.

In [43]:
from datasets import load_dataset
from transformer_lens.utils import tokenize_and_concatenate

dataset = load_dataset(
    path="NeelNanda/pile-10k",
    split="train",
    streaming=False,
)

token_dataset = tokenize_and_concatenate(
    dataset=dataset,
    tokenizer=tokenizer,
    streaming=True,
    max_length=128,
    add_bos_token=True,
)

# Run batches through the model and collect feature 6429 activations
top_activations = []  # list of (activation, token_str, context_tokens)
batch_size = 32

for batch_start in range(0, min(len(token_dataset), 5000), batch_size):
    batch_tokens = token_dataset[batch_start:batch_start + batch_size]["tokens"]
    texts = [tokenizer.decode(toks) for toks in batch_tokens]

    for j, text in enumerate(texts):
        with model.trace(text) as tracer:
            mlp_out = model.gpt_neox.layers[0].mlp.output.save()
        encoded = ae.encode(mlp_out)
        acts = encoded[0, :, 6429].detach().cpu()
        tokens = batch_tokens[j]
        for pos, a in enumerate(acts):
            if a.item() > 0:
                # store activation, token string, and surrounding context
                context_start = max(0, pos - 5)
                context_end = min(len(tokens), pos + 6)
                context = tokenizer.decode(tokens[context_start:context_end])
                top_activations.append((a.item(), tokenizer.decode(tokens[pos]), context))

    if batch_start % (batch_size * 10) == 0:
        print(f"processed {batch_start} sequences...")

# Sort and show top 20
top_activations.sort(key=lambda x: x[0], reverse=True)
print("\nTop 20 max-activating tokens for Feature 6429:")
for rank, (act, tok, ctx) in enumerate(top_activations[:20]):
    print(f"  {rank+1:>2d}. {repr(tok):<15s}  activation = {act:.4f}  context: ...{ctx}...")

README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

data/train-00000-of-00001-4746b8785c874c(…):   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

processed 0 sequences...
processed 320 sequences...
processed 640 sequences...
processed 960 sequences...
processed 1280 sequences...
processed 1600 sequences...
processed 1920 sequences...
processed 2240 sequences...
processed 2560 sequences...
processed 2880 sequences...
processed 3200 sequences...
processed 3520 sequences...
processed 3840 sequences...
processed 4160 sequences...
processed 4480 sequences...
processed 4800 sequences...

Top 20 max-activating tokens for Feature 6429:
   1. ' Paris'         activation = 7.1032  context: ...**Eurostar trains from Paris or Brussels** Arrive...
   2. ' Paris'         activation = 7.1032  context: ..., Leicester, Leeds, Paris.

A **...
   3. ' Paris'         activation = 7.1032  context: ... Woburn Estate, Paris House is a pretty black...
   4. ' Paris'         activation = 7.1032  context: ... including Amsterdam, Glasgow, Paris and Dublin.

...
   5. ' Paris'         activation = 7.1032  context: ...uk/passages; Paris St; adult/child...


It looks like we've found the feature for Paris! This is a small 10k token version of hte pile. it looks like our top activating sequences all have the word paris in them. sequences that activate (much) lower include activations for other cities in france and the abbreviation "bourg" which is associated with multiple French cities (Strasbourg, Cherbourg)

Let's check if the model has missed any references to paris france? (all top activating examples include the token 'paris', now lets see if all sequences that include the token 'paris' are in the top activations)

## Visualization 2: Token-Level Activation Heatmap

Here we visualize where Feature 6429 activates across the tokens of individual prompts. A monosemantic feature should activate selectively — lighting up on tokens related to "Paris" and staying dark elsewhere.

In [ ]:
from circuitsvis.tokens import colored_tokens_multi

# Show feature 6429 activation per token for a few positive and negative prompts
sample_prompts = [
    ("positive", positive_examples_paris[0]),
    ("positive", positive_examples_paris[1]),
    ("positive", positive_examples_paris[10]),
    ("negative", negative_examples_paris[0]),
    ("negative", negative_examples_paris[5]),
    ("false positive", "Bordeaux is a major city in France")
]


for label, prompt in sample_prompts:
    with model.trace(prompt) as tracer:
        mlp_out = model.gpt_neox.layers[0].mlp.output.save()
    encoded = ae.encode(mlp_out)
    # print(encoded.shape) # B, T, V where v gives vocab size
    feature_acts = encoded[0, :, 6429].detach().cpu() 
    tokens = [tokenizer.decode(t) for t in tokenizer.encode(prompt)]
    print(f"\n[{label.upper()}] {prompt}")
    display(colored_tokens_multi(tokens, feature_acts.unsqueeze(0).T))

torch.Size([1, 13, 32768])

[POSITIVE] The Eiffel Tower is the most iconic landmark in Paris.


torch.Size([1, 10, 32768])

[POSITIVE] Paris is the capital and largest city of France.


torch.Size([1, 14, 32768])

[POSITIVE] Hemingway once wrote that Paris is a moveable feast.


torch.Size([1, 12, 32768])

[NEGATIVE] The Colosseum is the most iconic landmark in Rome.


torch.Size([1, 14, 32768])

[NEGATIVE] He proposed to her on a bridge over the Tiber in Rome.


torch.Size([1, 9, 32768])

[FALSE POSITIVE] Bordeaux is a major city in France


## Visualization 3: Activation Distribution — Positive vs Negative Prompts

A well-separated distribution is strong evidence of monosemanticity. If the feature activates strongly and consistently on Paris prompts but stays near zero on non-Paris prompts, it behaves like a reliable detector for a single concept.

In [31]:
import plotly.graph_objects as go

# Compute mean activation of feature 6429 for each prompt
pos_acts = [features_dict[f'positive_{i}']['features'][..., 6429].mean().item() for i in range(50)]
neg_acts = [features_dict[f'negative_{i}']['features'][..., 6429].mean().item() for i in range(50)]

fig = go.Figure()
fig.add_trace(go.Histogram(x=pos_acts, name='Positive (Paris)', opacity=0.7, marker_color='blue'))
fig.add_trace(go.Histogram(x=neg_acts, name='Negative (non-Paris)', opacity=0.7, marker_color='red'))
fig.update_layout(
    barmode='overlay',
    title='Feature 6429: Activation Distribution',
    xaxis_title='Mean Activation',
    yaxis_title='Count',
    plot_bgcolor='white',
)
fig.show()

Based on this, feature 6429 Very clearly seems to be a monosemantic feature for Paris, France. Look at its activation difference, 10x the second largest feature! Let's do the following:

1. calculate the specificity and sensitivity as detailed in the anthropic whitepaper. 
2. use SAELens and Logit Lens to produce some cool visualizations
3. perform a causal intervention where we permanently acitvate this feature. 



Let's calculate specificity and sensitivity (or at least estimate them)
Then lets do a causal intervention where we permanently activate feature like 10x and see if we get something like golden gate claude